# SC-4-Functions-State - Fonctions et Etat

[<< Solidity Basics](SC-3-Solidity-Basics.ipynb) | [Inheritance >>](SC-5-Inheritance.ipynb)

---

## Objectifs d'apprentissage

1. Comprendre les **data locations** (storage, memory, calldata)
2. Maitriser la **visibilite** des fonctions
3. Utiliser les **modifiers** (view, pure, payable)

### Prerequis

- [SC-3-Solidity-Basics](SC-3-Solidity-Basics.ipynb) complete
- Notions de base Solidity (types, variables d etat)
- Remix IDE ou environnement local configure

### Duree estimee : 45 minutes

---

## 0. Connexion a la blockchain locale

Tous les contrats de ce notebook sont **compiles et deployes reellement** sur anvil.
Lancez `anvil` dans un terminal avant d'executer les cellules.


In [ ]:
# Connection a anvil (blockchain locale Foundry)
# Prerequis: anvil en cours d'execution dans un terminal
from web3 import Web3
import solcx

SOLC_VERSION = "0.8.28"
ANVIL_URL = "http://127.0.0.1:8545"

# Connexion
w3 = Web3(Web3.HTTPProvider(ANVIL_URL))
assert w3.is_connected(), f"Impossible de se connecter a {ANVIL_URL}. Lancez 'anvil' dans un terminal."

# Installer solc si necessaire
installed = [str(v) for v in solcx.get_installed_solc_versions()]
if SOLC_VERSION not in installed:
    solcx.install_solc(SOLC_VERSION)
solcx.set_solc_version(SOLC_VERSION)

deployer = w3.eth.accounts[0]
print(f"Connecte a anvil (chain {w3.eth.chain_id}), deployer: {deployer[:10]}...")


def compile_and_deploy(w3, source_code, deployer, *constructor_args):
    """Compiler et deployer un contrat Solidity."""
    compiled = solcx.compile_source(
        source_code, output_values=["abi", "bin"], solc_version=SOLC_VERSION
    )
    contract_id, contract_interface = compiled.popitem()
    Contract = w3.eth.contract(
        abi=contract_interface["abi"], bytecode=contract_interface["bin"]
    )
    tx_hash = Contract.constructor(*constructor_args).transact({"from": deployer})
    receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
    instance = w3.eth.contract(
        address=receipt.contractAddress, abi=contract_interface["abi"]
    )
    print(f"Deploye: {contract_id.split(':')[-1]} a {receipt.contractAddress}")
    return instance, receipt


---

## 1. Data Locations

Solidite definit trois emplacements de donnees :

| Location | Description | Cout gas |
|----------|-------------|----------|
| `storage` | Blockchain permanente | Eleve |
| `memory` | Temporaire (RAM) | Faible |
| `calldata` | Donnees d'entree (read-only) | Minimal |

In [1]:
# Data locations en Solidity
DATA_LOCATION_EXAMPLE = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract DataLocations {
    // STORAGE : stocke sur la blockchain (permanent)
    uint256[] public numbers;  // storage par defaut pour variables d'etat
    mapping(address => uint256) public balances;

    // MEMORY : temporaire, efface apres l'appel
    function processArray(uint256[] memory arr) public pure returns (uint256) {
        uint256 sum = 0;
        for (uint i = 0; i < arr.length; i++) {
            sum += arr[i];
        }
        return sum;
    }

    // CALLDATA : read-only, le moins couteux
    function getElement(uint256[] calldata arr, uint256 index) 
        external pure returns (uint256) 
    {
        return arr[index];
    }

    // Storage reference pour modification
    function updateNumber(uint256 index, uint256 value) public {
        numbers[index] = value;  // Modification directe du storage
    }
}
'''


# Compilation et deploiement reel sur anvil
datalocations, receipt = compile_and_deploy(w3, DATA_LOCATION_EXAMPLE, deployer)

Data Locations en Solidity:

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract DataLocations {
    // STORAGE : stocke sur la blockchain (permanent)
    uint256[] public numbers;  // storage par defaut pour variables d'etat
    mapping(address => uint256) public balances;

    // MEMORY : temporaire, efface apres l'appel
    function processArray(uint256[] memory arr) public pure returns (uint256) {
        uint256 sum = 0;
        for (uint i = 0; i < arr.length; i++) {
            sum += arr[i];
        }
        return sum;
    }

    // CALLDATA : read-only, le moins couteux
    function getElement(uint256[] calldata arr, uint256 index) 
        external pure returns (uint256) 
    {
        return arr[index];
    }

    // Storage reference pour modification
    function updateNumber(uint256 index, uint256 value) public {
        numbers[index] = value;  // Modification directe du storage
    }
}



### 1.1 Regles de data location

- **Variables d'etat** : toujours en `storage`
- **Arguments de fonction** : `memory` ou `calldata` pour les types complexes
- **Variables locales** : `storage` (reference) ou `memory` (valeur copiee)

In [ ]:
# Storage vs Memory : demonstration de la difference critique
STORAGE_MEMORY_EXAMPLE = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract StorageVsMemory {
    struct User {
        string name;
        uint256 balance;
    }

    User[] public users;

    function addUser(string memory name, uint256 balance) public {
        users.push(User(name, balance));
    }

    // STORAGE reference : modifie directement le tableau
    function updateBalanceGood(uint256 index, uint256 amount) public {
        User storage user = users[index];
        user.balance = amount;  // Modification permanente
    }

    // MEMORY copy : ne modifie PAS le storage (bug courant!)
    function updateBalanceBad(uint256 index, uint256 amount) public {
        User memory user = users[index];
        user.balance = amount;  // Modification locale uniquement!
    }

    function getBalance(uint256 index) public view returns (uint256) {
        return users[index].balance;
    }
}
'''

print("--- Storage vs Memory : difference critique ---")
sm_contract, receipt = compile_and_deploy(w3, STORAGE_MEMORY_EXAMPLE, deployer)
print(f"  Deploye : {sm_contract.address}")

# Ajouter un utilisateur
sm_contract.functions.addUser("Alice", 1000).transact({'from': deployer})
print(f"  Alice balance initiale : {sm_contract.functions.getBalance(0).call()}")

# Mettre a jour via storage (correct)
sm_contract.functions.updateBalanceGood(0, 2000).transact({'from': deployer})
print(f"  Apres updateBalanceGood(2000) : {sm_contract.functions.getBalance(0).call()} (attendu: 2000)")

# Mettre a jour via memory (bug - ne change rien)
sm_contract.functions.updateBalanceBad(0, 9999).transact({'from': deployer})
print(f"  Apres updateBalanceBad(9999) : {sm_contract.functions.getBalance(0).call()} (attendu: 2000, pas 9999!)")

---

## 2. Visibilite des fonctions

| Visibilite | Exterieur | Contrat enfant | Meme contrat |
|------------|-----------|----------------|--------------|
| `public` | Oui | Oui | Oui |
| `external` | Oui | Non | Non |
| `internal` | Non | Oui | Oui |
| `private` | Non | Non | Oui |

In [ ]:
# Visibilite des fonctions - demonstration
VISIBILITY_EXAMPLE = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract VisibilityExample {
    uint256 private privateVar = 1;
    uint256 internal internalVar = 2;
    uint256 public publicVar = 3;

    function publicFunction() public pure returns (string memory) {
        return "Je suis publique!";
    }

    function externalFunction() external pure returns (string memory) {
        return "Je suis externe!";
    }

    function internalFunction() internal pure returns (string memory) {
        return "Je suis interne!";
    }

    function privateFunction() private pure returns (string memory) {
        return "Je suis prive!";
    }

    function callInternal() public pure returns (string memory) {
        return internalFunction();
    }

    function getPrivateVar() public view returns (uint256) {
        return privateVar;
    }

    function getAllVars() public view returns (uint256, uint256, uint256) {
        return (privateVar, internalVar, publicVar);
    }
}
'''

print("--- Visibilite : public / external / internal / private ---")
vis_contract, receipt = compile_and_deploy(w3, VISIBILITY_EXAMPLE, deployer)
print(f"  Deploye : {vis_contract.address}")
print(f"  publicFunction() = '{vis_contract.functions.publicFunction().call()}'")
print(f"  callInternal() = '{vis_contract.functions.callInternal().call()}'")
print(f"  publicVar = {vis_contract.functions.publicVar().call()}")
priv, intern, pub = vis_contract.functions.getAllVars().call()
print(f"  getAllVars() : private={priv}, internal={intern}, public={pub}")

In [ ]:
# External vs Public : optimisation gas
EXTERNAL_OPTIMIZATION = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract GasOptimization {
    // EXTERNAL est moins couteux pour les tableaux
    // car les donnees restent en calldata (pas de copie)

    // Moins efficace (copie en memory)
    function sumPublic(uint256[] memory arr) public pure returns (uint256) {
        uint256 total = 0;
        for (uint i = 0; i < arr.length; i++) {
            total += arr[i];
        }
        return total;
    }

    // Plus efficace (reste en calldata)
    function sumExternal(uint256[] calldata arr) external pure returns (uint256) {
        uint256 total = 0;
        for (uint i = 0; i < arr.length; i++) {
            total += arr[i];
        }
        return total;
    }
}
'''

print("--- External vs Public : optimisation gas ---")
gas_contract, receipt = compile_and_deploy(w3, EXTERNAL_OPTIMIZATION, deployer)
print(f"  Deploye : {gas_contract.address}")

# Tableau de test : [1, 2, 3, 4, 5]
arr = [1, 2, 3, 4, 5]
result_public = gas_contract.functions.sumPublic(arr).call()
result_external = gas_contract.functions.sumExternal(arr).call()
print(f"  sumPublic([1..5])   = {result_public}  (memory copy)")
print(f"  sumExternal([1..5]) = {result_external}  (calldata, plus efficace)")
print("  Resultat identique, mais external consomme moins de gas pour les grands tableaux")

---

## 3. Modificateurs de fonction

### 3.1 view et pure

In [ ]:
# view et pure
VIEW_PURE_EXAMPLE = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract ViewPureExample {
    uint256 public value = 42;

    // VIEW : lit le storage mais ne modifie pas
    function getValue() public view returns (uint256) {
        return value;
    }

    // PURE : ni lecture ni ecriture du storage
    function add(uint256 a, uint256 b) public pure returns (uint256) {
        return a + b;
    }

    // Fonction normale : peut lire et ecrire
    function increment() public returns (uint256) {
        value += 1;
        return value;
    }
}
'''

print("--- view = lecture seule, pure = sans acces storage ---")
vp_contract, receipt = compile_and_deploy(w3, VIEW_PURE_EXAMPLE, deployer)
print(f"  Deploye : {vp_contract.address}")
print(f"  getValue() = {vp_contract.functions.getValue().call()}  (view : lit value=42)")
print(f"  add(10, 32) = {vp_contract.functions.add(10, 32).call()}  (pure : calcul sans storage)")
vp_contract.functions.increment().transact({'from': deployer})
print(f"  Apres increment() : getValue() = {vp_contract.functions.getValue().call()}  (attendu: 43)")

### 3.2 payable

In [ ]:
# payable pour recevoir des ETH
PAYABLE_EXAMPLE = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract PayableExample {
    address public owner;
    mapping(address => uint256) public balances;

    constructor() {
        owner = msg.sender;
    }

    // Fonction payable : peut recevoir des ETH
    function deposit() public payable {
        balances[msg.sender] += msg.value;
    }

    // Retirer des ETH
    function withdraw(uint256 amount) public {
        require(balances[msg.sender] >= amount, "Fonds insuffisants");
        balances[msg.sender] -= amount;
        (bool success, ) = payable(msg.sender).call{value: amount}("");
        require(success, "Echec du transfert");
    }

    // Consulter le solde du contrat
    function getContractBalance() public view returns (uint256) {
        return address(this).balance;
    }

    // Receive : appele lors d'un transfert ETH simple
    receive() external payable {
        balances[msg.sender] += msg.value;
    }
}
'''

print("--- payable : recevoir et retirer des ETH ---")
pay_contract, receipt = compile_and_deploy(w3, PAYABLE_EXAMPLE, deployer)
print(f"  Deploye : {pay_contract.address}")
print(f"  Balance initiale du contrat : {pay_contract.functions.getContractBalance().call()} wei")

# Deposer 1 ETH (1e18 wei)
one_eth = w3.to_wei(1, 'ether')
pay_contract.functions.deposit().transact({'from': deployer, 'value': one_eth})
print(f"  Apres deposit(1 ETH) : contrat = {w3.from_wei(pay_contract.functions.getContractBalance().call(), 'ether')} ETH")
print(f"  Solde deployer dans contrat : {w3.from_wei(pay_contract.functions.balances(deployer).call(), 'ether')} ETH")

# Retirer 0.5 ETH
half_eth = w3.to_wei(0.5, 'ether')
pay_contract.functions.withdraw(half_eth).transact({'from': deployer})
print(f"  Apres withdraw(0.5 ETH) : contrat = {w3.from_wei(pay_contract.functions.getContractBalance().call(), 'ether')} ETH")

---

## 4. Custom Modifiers

Les modifiers permettent de reutiliser des conditions prealables.

In [ ]:
# Custom modifiers
MODIFIER_EXAMPLE = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract ModifierExample {
    address public owner;
    bool public paused = false;
    uint256 public counter = 0;

    constructor() {
        owner = msg.sender;
    }

    modifier onlyOwner() {
        require(msg.sender == owner, "Pas le proprietaire");
        _;
    }

    modifier whenNotPaused() {
        require(!paused, "Contrat en pause");
        _;
    }

    function pause() public onlyOwner {
        paused = true;
    }

    function unpause() public onlyOwner {
        paused = false;
    }

    function increment() public whenNotPaused {
        counter += 1;
    }

    function criticalOperation() public onlyOwner whenNotPaused returns (uint256) {
        counter += 10;
        return counter;
    }
}
'''

print("--- Custom modifiers : onlyOwner + whenNotPaused ---")
mod_contract, receipt = compile_and_deploy(w3, MODIFIER_EXAMPLE, deployer)
print(f"  Deploye : {mod_contract.address}")
print(f"  paused = {mod_contract.functions.paused().call()}, counter = {mod_contract.functions.counter().call()}")

# Incrementer (fonctionne car pas en pause)
mod_contract.functions.increment().transact({'from': deployer})
print(f"  Apres increment() : counter = {mod_contract.functions.counter().call()}")

# Mettre en pause
mod_contract.functions.pause().transact({'from': deployer})
print(f"  Apres pause() : paused = {mod_contract.functions.paused().call()}")

# increment() doit revert
try:
    mod_contract.functions.increment().transact({'from': deployer})
    print("  ERREUR : increment() aurait du revert!")
except Exception as e:
    print(f"  increment() reverte (attendu) : Contrat en pause")

# Reprendre et faire criticalOperation
mod_contract.functions.unpause().transact({'from': deployer})
result = mod_contract.functions.criticalOperation().transact({'from': deployer})
print(f"  Apres unpause() + criticalOperation() : counter = {mod_contract.functions.counter().call()}")

---

## 5. Fonctions speciales

In [ ]:
# Constructor, receive, fallback
SPECIAL_FUNCTIONS = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract SpecialFunctions {
    address public owner;
    string public name;
    uint256 public receivedTotal = 0;

    event Received(address from, uint256 amount);

    // CONSTRUCTOR : execute une seule fois au deploiement
    constructor(string memory _name) {
        owner = msg.sender;
        name = _name;
    }

    // RECEIVE : appele lors d'un transfert ETH simple (sans data)
    receive() external payable {
        receivedTotal += msg.value;
        emit Received(msg.sender, msg.value);
    }

    function getBalance() public view returns (uint256) {
        return address(this).balance;
    }
}
'''

print("--- Fonctions speciales : constructor, receive ---")
# Constructor prend un argument string "_name"
special, receipt = compile_and_deploy(w3, SPECIAL_FUNCTIONS, deployer, "MonContrat")
print(f"  Deploye : {special.address}")
print(f"  name = '{special.functions.name().call()}'  (initialise par constructor)")
print(f"  owner = {special.functions.owner().call()[:10]}...")
print(f"  receivedTotal initial = {special.functions.receivedTotal().call()} wei")

# Envoyer des ETH via receive() (transfert simple sans data)
w3.eth.send_transaction({'from': deployer, 'to': special.address, 'value': w3.to_wei(0.1, 'ether')})
print(f"  Apres transfert 0.1 ETH : receivedTotal = {w3.from_wei(special.functions.receivedTotal().call(), 'ether')} ETH")
print(f"  Balance contrat : {w3.from_wei(special.functions.getBalance().call(), 'ether')} ETH")

---

## 6. Exercices

### Exercice 1 : Contrat Bank

Creez un contrat qui permet de deposer et retirer des ETH avec un modifier `onlyOwner`.

In [ ]:
# Exercice 1 : Contrat Bank
EXERCICE_BANK = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract Bank {
    address public owner;
    mapping(address => uint256) public balances;

    constructor() {
        // TODO: Initialiser le proprietaire
    }

    modifier onlyOwner() {
        // TODO: Verifier que msg.sender est le proprietaire
        // TODO: Utiliser require et ne pas oublier le _;
        _;
    }

    function deposit() public payable {
        // TODO: Ajouter msg.value au solde de msg.sender
    }

    function withdraw(uint256 amount) public {
        // TODO: Verifier que l'utilisateur a assez de fonds
        // TODO: Reduire le solde de l'utilisateur
        // TODO: Transferer les ETH a l'utilisateur
    }

    function emergencyWithdraw() public onlyOwner {
        // TODO: Transferer tout le solde du contrat au proprietaire
    }
}
'''

# Compilation et deploiement reel sur anvil
bank, receipt = compile_and_deploy(w3, EXERCICE_BANK, deployer)

### Exercice 2 : Contrat Pausable

Creez un contrat avec des fonctions qui peuvent etre mises en pause.

In [ ]:
# Exercice 2 : Contrat Pausable
EXERCICE_PAUSABLE = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract Pausable {
    address public owner;
    bool public paused = false;
    uint256 public counter = 0;

    constructor() {
        // TODO: Initialiser le proprietaire
    }

    modifier onlyOwner() {
        // TODO: Verifier que msg.sender est le proprietaire
        _;
    }

    modifier whenNotPaused() {
        // TODO: Verifier que le contrat n est pas en pause
        _;
    }

    function pause() public onlyOwner {
        // TODO: Mettre le contrat en pause
    }

    function unpause() public onlyOwner {
        // TODO: Reprendre le contrat
    }

    function increment() public whenNotPaused {
        // TODO: Incrementer le compteur
    }

    function reset() public onlyOwner whenNotPaused {
        // TODO: Remettre le compteur a zero
    }
}
'''

# Compilation et deploiement reel sur anvil
pausable, receipt = compile_and_deploy(w3, EXERCICE_PAUSABLE, deployer)

---

## 7. Resume

| Concept | Description |
|---------|-------------|
| `storage` | Donnees permanentes sur blockchain |
| `memory` | Donnees temporaires en RAM |
| `calldata` | Donnees d'entree read-only |
| `public` | Accessible partout |
| `external` | Accessible uniquement de l'exterieur |
| `internal` | Ce contrat et enfants |
| `private` | Uniquement ce contrat |
| `view` | Lecture seule du storage |
| `pure` | Aucun acces au storage |
| `payable` | Peut recevoir des ETH |

---

**Notebook suivant** : [SC-5-Inheritance](SC-5-Inheritance.ipynb)

---

[<< Solidity Basics](SC-3-Solidity-Basics.ipynb) | [Inheritance >>](SC-5-Inheritance.ipynb)